In [45]:
import ollama
import os
import re
import pandas as pd
from tqdm import tqdm
import ast
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import spacy


In [46]:
# Load English tokenizer, tagger, parser and NER
nlp = spacy.load("en_core_web_sm")

In [47]:
#Read in our genre list
genres = pd.read_csv('genres_taxonomy_quick.csv')  # Replace 'genres_taxonomy_quick.csv' with the actual file name
genres = genres['genre']
#Read in our movie data
movies = pd.read_csv("netflix_titles.csv")
movies = movies.sample(n=1000) #This takes a while to run so I didn't do it for the entire dataset at once

In [48]:
# OLLAMA_BASE_URL in .env is the OpenAI-compat endpoint (ends in /v1); the ollama client needs the native API root.
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").removesuffix("/v1")
ollama_client = ollama.Client(host=OLLAMA_BASE_URL)
# gpt-oss:20b is a reasoning model that spends its token budget on hidden "thinking" before answering,
# leaving nothing for the actual response within a short num_predict. llama3.2 has no thinking mode,
# so it answers directly and fits this short classification task.
OLLAMA_MODEL = "llama3.2:latest"

In [49]:
def predict_genres(movie_description):
    prompt = f"Predict the top three genres (only genres, not descriptions) for a movie with the following description: {movie_description}"
    response = ollama_client.generate(
        model=OLLAMA_MODEL,
        prompt=prompt,
        options={"num_predict": 50, "temperature": 0.2},
    )
    predicted_genres = response.response.strip()
    return predicted_genres

In [50]:
def filter_predicted_genres(predicted_genres, predefined_genres):
    # Use word embeddings to calculate semantic similarity between predicted and predefined genres
    predicted_genres_tokens = nlp(predicted_genres)
    predicted_genres_tokens = predicted_genres_tokens.text
    # Use regular expression to extract genres
    genres_with_numbers = re.findall(r'\d+\.\s*([^\n]+)', predicted_genres_tokens)
    # Remove leading/trailing whitespaces from each genre
    predicted_genres = [genre.strip().lower() for genre in genres_with_numbers]
    filtered_genres = []
    similarity_scores = []
    for predicted_genre in predicted_genres:
        max_similarity = 0
        best_match = None
        for predefined_genre in predefined_genres:
            similarity_score = nlp(predicted_genre).similarity(nlp(predefined_genre))
            if similarity_score > max_similarity:  # Adjust the threshold as needed
                max_similarity = similarity_score
                best_match = predefined_genre
        filtered_genres.append(best_match)
        similarity_scores.append(max_similarity)
    # Sort the filtered genres based on the similarity scores
    filtered_genres = [x for _, x in sorted(zip(similarity_scores, filtered_genres), reverse=True)]    
    return filtered_genres

In [51]:
# pandas 3.x defaults bare strings to the new "str" dtype, which can't hold a list per cell.
# Use object dtype explicitly since each cell will later hold a list of genres.
movies['predicted_genres'] = pd.Series([""] * len(movies), index=movies.index, dtype="object")

In [52]:
def add_predicted_genres_to_df(df, predefined_genres):   
    # Iterate through the dataframe
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        # Apply the predict_genres function to the movie description
        predicted_genres = predict_genres(row['description'])
        # Prioritize the predicted genres
        filtered_genres = filter_predicted_genres(predicted_genres, predefined_genres)
        # Add the prioritized genres to the dataframe
        df.at[index, 'predicted_genres'] = filtered_genres
add_predicted_genres_to_df(movies, genres)

  0%|          | 0/1000 [00:00<?, ?it/s]/var/folders/ls/grnwd6y54m58zjpyhg_hg8880000gn/T/ipykernel_42326/1444019431.py:15: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  similarity_score = nlp(predicted_genre).similarity(nlp(predefined_genre))
100%|██████████| 1000/1000 [35:02<00:00,  2.10s/it]


In [53]:
# Split the lists into separate columns with specific names
movies[['genre1', 'genre2', 'genre3']] = movies['predicted_genres'].apply(lambda x: pd.Series((x + [None, None, None])[:3]))

In [54]:
#I have the tags saved so I don't have to run it again
#movies = pd.read_csv("moviesWithTags.csv")
movies.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,predicted_genres,genre1,genre2,genre3
6132,s6133,TV Show,America's Book of Secrets,NaN,Jonathan Adams,United States,"September 15, 2019",2014,TV-14,1 Season,Docuseries,This engaging documentary series shares the su...,"[mystery, history, documentary]",mystery,history,documentary
3326,s3327,Movie,A Remarkable Tale,Marina Seresesky,"Carmen Machi, Pepón Nieto, Kiti Mánver, Jon Ko...",Spain,"November 1, 2019",2019,TV-MA,94 min,"Comedies, International Movies",The residents of an isolated town look to revi...,"[drama, comedy, social/activist]",drama,comedy,social/activist
3968,s3969,TV Show,The Terminal,NaN,"Mohammad Mansour, Bouthayna Al Raesi, Bassima ...",NaN,"March 29, 2019",2018,TV-14,1 Season,"International TV Shows, TV Dramas","Social pressure, a complicated past and a clas...","[thriller, family, drama]",thriller,family,drama
3515,s3516,TV Show,Surviving R. Kelly,NaN,R. Kelly,United States,"September 15, 2019",2019,TV-MA,1 Season,"Crime TV Shows, Docuseries",In this documentary series on the tangled hist...,"[true crime, drama, documentary]",true crime,drama,documentary
8708,s8709,Movie,We're No Animals,Alejandro Agresti,"John Cusack, Paul Hipp, Kevin Morris, Alejandr...","United States, Argentina","August 15, 2017",2015,TV-MA,94 min,"Comedies, Dramas, Independent Movies","Unhappy with his commercial film work, a jaded...","[thriller, mystery, drama]",thriller,mystery,drama


In [55]:
#Keep only the columns we need for similarity
movies = movies[['title','genre1','genre2','genre3']]
#Drop duplicates
movies = movies.drop_duplicates()
#Set the 'title' column as our index
movies = movies.set_index('title')
# Stash genre1/genre2/genre3 so we can add them back to the CSV later; the similarity
# pipeline below drops them (it only works with the numeric one-hot dummy columns).
genre_cols_for_export = movies[['genre1', 'genre2', 'genre3']].copy()

In [56]:
# Combine genre columns into a single column
# fillna before astype(str): pandas 3.x leaves real NaN/None as floats through astype(str)
# instead of stringifying them to "nan", which breaks ','.join on rows with missing genres.
movies['all_genres'] = movies[['genre1', 'genre2', 'genre3']].fillna('').astype(str).agg(','.join, axis=1)
# Split the genres and create dummy variables for each genre
genres = movies['all_genres'].str.get_dummies(sep=',')
# Concatenate the dummy variables with the original DataFrame
movies = pd.concat([movies, genres], axis=1)
# Drop unnecessary columns
movies.drop(['all_genres', 'genre1', 'genre2', 'genre3'], axis=1, inplace=True)

In [57]:
# If there are duplicate columns due to the one-hot encoding, you can sum them up
# pandas 3.x removed groupby(axis=1) entirely; transpose, group on the index, then transpose back.
movie_genre_matrix = movies.T.groupby(level=0).sum().T
# Calculate cosine similarity 
similarity_matrix = cosine_similarity(movie_genre_matrix, movie_genre_matrix)

In [58]:
def find_similar_movies(movie_name, movie_genre_matrix, num_similar_movies=3):
    # Calculate cosine similarity
    similarity_matrix = cosine_similarity(movie_genre_matrix, movie_genre_matrix)    
    # Find the index of the given movie
    movie_index = movie_genre_matrix.index.get_loc(movie_name)    
    # Sort and get indices of most similar movies (excluding the movie itself)
    most_similar_indices = np.argsort(similarity_matrix[movie_index])[:-num_similar_movies-1:-1]    
    # Return the most similar movies
    return movie_genre_matrix.index[most_similar_indices].tolist()

In [61]:
movies = movies.reset_index()
movie_genre_matrix.index[:10].tolist()  # see what's available

["America's Book of Secrets",
 'A Remarkable Tale',
 'The Terminal',
 'Surviving R. Kelly',
 "We're No Animals",
 'Miracle',
 'Calum von Moger: Unbroken',
 'Life in Color with David Attenborough',
 "Ojos in d' House",
 'We the Animals']

In [72]:
# Example usage
similar_movies = find_similar_movies("Ojos in d' House", movie_genre_matrix, num_similar_movies=4)
print(similar_movies)

['Something Huge', 'Cristela Alonzo: Lower Classy', 'Off Course', 'The Ingenuity of the Househusband']


In [64]:
movie_title = 'Calum von Moger: Unbroken'
movies.loc[movies['title'] == movie_title].groupby('title').sum().T.sort_values(by=movie_title,ascending=False).head(3)

title,Calum von Moger: Unbroken
index,6
inspirational,1
sports,1


In [71]:
movie_title = 'Surviving R. Kelly'
movies.loc[movies['title'] == movie_title].groupby('title').sum().T.sort_values(by=movie_title,ascending=False).head(3)

title,Surviving R. Kelly
index,3
true crime,1
drama,1


In [68]:
# Example usage
similar_movies = find_similar_movies("Calum von Moger: Unbroken", movie_genre_matrix, num_similar_movies=4)
print(similar_movies)

['Soorma', 'Calum von Moger: Unbroken', 'The Brawler', '3 Heroines']


In [73]:
similar_movies = find_similar_movies("Life in Color with David Attenborough", movie_genre_matrix, num_similar_movies=4)
print(similar_movies)

['Life in Color with David Attenborough', 'The Blue Planet: A Natural History of the Oceans', 'F is for Family', 'Birders']


In [75]:
movie_title = "Life in Color with David Attenborough"
movies.loc[movies['title'] == movie_title].groupby('title').sum().T.sort_values(by=movie_title,ascending=False).head(3)

title,Life in Color with David Attenborough
index,7
wildlife,1
period piece,1


In [76]:
# Add genre1/genre2/genre3 back (stashed earlier) so the visualization section can melt on them
movies = movies.merge(genre_cols_for_export.reset_index(), on='title', how='left')
movies.to_csv("moviesWithTags.csv")

# Visualization code

In [77]:
import pandas as pd
from itertools import combinations

In [78]:
movies = pd.read_csv("moviesWithTags.csv")

In [79]:
# Melt the dataframe to unpivot genre columns
melted_df = pd.melt(movies, id_vars=['title'], value_vars=['genre1', 'genre2', 'genre3'], var_name='Genre', value_name='GenreValue')
genre_links = pd.crosstab(index=melted_df['title'], columns=melted_df['GenreValue'])
# Create combinations of genres for each title
combinations_list = []
for title, group in melted_df.groupby('title')['GenreValue']:
    genre_combinations = list(combinations(group, 2))
    combinations_list.extend([(title, combo[0], combo[1]) for combo in genre_combinations])
# Create a new dataframe from the combinations list
combinations_df = pd.DataFrame(combinations_list, columns=['title', 'Genre1', 'Genre2'])
combinations_df = combinations_df[['Genre1','Genre2']]
combinations_df = combinations_df.rename(columns={"Genre1": "source", "Genre2": "target"}, errors="raise")
combinations_df = combinations_df.set_index('source')
combinations_df.to_csv("genreCombos.csv")
combinations_df.to_pickle("genreCombos.pkl")